# Previsão de Vendas Diárias — Próximos 7 dias

Este notebook adapta o pipeline de análise preditiva (originalmente mensal, sazonalidade de 12 meses) para dados **diários**, com sazonalidade **semanal** (período = 7 dias). O objetivo é prever o valor de `venda` para os próximos 7 dias.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
warnings.filterwarnings('ignore')

## 1. Carregando e limpando os dados

O arquivo de origem usa vírgula como separador de colunas, mas os valores numéricos vêm entre aspas com vírgula como separador decimal (padrão brasileiro), por exemplo: `"9004,17"`. Além disso, o arquivo tem linhas de "lixo" no final (resíduo de fórmulas do Excel/Sheets, sem data e com `mes/ano` = `12/1899`), que precisam ser descartadas.

In [ ]:
# Ajuste o caminho abaixo para onde estiver o seu CSV
CAMINHO_CSV = 'dados/relatorio_vendas.csv'

df = pd.read_csv(CAMINHO_CSV)
df.head()

In [ ]:
def to_float_br(series):
    """Converte string numérica no formato brasileiro ('1.234,56') para float."""
    return (series.astype(str)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .astype(float))

# Remove linhas sem data (lixo no final do arquivo)
df = df.dropna(subset=['data']).copy()

# Converte a data (formato dia/mes/ano)
df['data'] = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['data'])

# Converte colunas numéricas no formato brasileiro
colunas_numericas = ['venda', 'valor_cancelado', 'valor_total', 'ticket_medio']
for col in colunas_numericas:
    df[col] = to_float_br(df[col])

# Ordena cronologicamente e define a data como índice com frequência diária
df = df.sort_values('data').set_index('data')
df = df.asfreq('D')

print('Formato final:', df.shape)
df.head()

In [ ]:
# Verificando se sobrou algum dia sem valor de venda (buraco na série)
print('Dias sem valor de venda:', df['venda'].isnull().sum())

# Caso existam dias faltantes, interpola linearmente para não quebrar o modelo
df['venda'] = df['venda'].interpolate()

## 2. Análise Exploratória (EDA)

In [ ]:
# Resumo estatístico
df['venda'].describe()

In [ ]:
# Verificando outliers
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['venda'])
plt.title('Distribuição de Vendas Diárias — Outliers')
plt.show()

In [ ]:
# Série histórica completa
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['venda'])
plt.title('Vendas Diárias — Série Histórica')
plt.xlabel('Data')
plt.ylabel('Venda (R$)')
plt.show()

## 3. Decomposição da série (sazonalidade semanal)

Diferente do modelo mensal (período = 12), aqui usamos **período = 7** para capturar o padrão que se repete a cada semana (ex: fins de semana com mais ou menos movimento).

In [ ]:
decomposicao = seasonal_decompose(df['venda'], model='additive', period=7)

plt.figure(figsize=(12, 8))

plt.subplot(411)
plt.plot(df['venda'], label='Série Original')
plt.legend(loc='best')

plt.subplot(412)
plt.plot(decomposicao.trend, label='Tendência')
plt.legend(loc='best')

plt.subplot(413)
plt.plot(decomposicao.seasonal, label='Sazonalidade Semanal')
plt.legend(loc='best')

plt.subplot(414)
plt.plot(decomposicao.resid, label='Resíduo')
plt.legend(loc='best')

plt.tight_layout()
plt.show()

## 4. Teste de Estacionariedade (Dickey-Fuller Aumentado)

In [ ]:
def teste_estacionariedade(timeseries, nome='série'):
    print(f'Resultado do teste Dickey-Fuller para {nome}:')
    dfteste = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dfteste[0:4], index=['Teste estatístico',
                                               'p-value',
                                               '#Lags Usados',
                                               'Número de observações Usadas'])
    for key, value in dfteste[4].items():
        dfoutput['Valor Crítico (%s)' % key] = value
    print(dfoutput)
    print()

teste_estacionariedade(df['venda'], 'venda (original)')

In [ ]:
# Diferenciação (para efeito de comparação/diagnóstico)
df['venda_diff'] = df['venda'].diff()
df_diff = df.dropna(subset=['venda_diff'])

teste_estacionariedade(df_diff['venda_diff'], 'venda (diferenciada)')

Se o p-value da série original já for menor que 0,05, a série já é estatisticamente estacionária — mas isso não elimina a sazonalidade semanal, que o SARIMAX trata separadamente através da diferenciação sazonal (`D=1`, período 7).

## 5. Comparação de modelos

Testamos algumas combinações de ordem nao-sazonal (p, d, q) mantendo a componente sazonal semanal fixa `(1, 1, 1, 7)`, e escolhemos o modelo com menor AIC.

In [ ]:
candidatos = [(1, 0, 1), (1, 1, 1), (2, 0, 1)]
resultados = []

for ordem in candidatos:
    modelo = SARIMAX(df['venda'], order=ordem, seasonal_order=(1, 1, 1, 7),
                      enforce_stationarity=False, enforce_invertibility=False)
    modelo_treinado = modelo.fit(disp=False)
    resultados.append((ordem, modelo_treinado.aic, modelo_treinado.bic))
    print(f'Ordem {ordem} -> AIC: {modelo_treinado.aic:.2f} | BIC: {modelo_treinado.bic:.2f}')

melhor_ordem = min(resultados, key=lambda x: x[1])[0]
print(f'\nMelhor ordem não-sazonal pelo critério AIC: {melhor_ordem}')

## 6. Modelo Final — SARIMAX

In [ ]:
ORDEM_NAO_SAZONAL = melhor_ordem   # escolhido automaticamente pela etapa anterior
ORDEM_SAZONAL = (1, 1, 1, 7)        # sazonalidade semanal

modelo_final = SARIMAX(df['venda'], order=ORDEM_NAO_SAZONAL, seasonal_order=ORDEM_SAZONAL,
                        enforce_stationarity=False, enforce_invertibility=False)
modelo_treinado = modelo_final.fit(disp=False)
print(modelo_treinado.summary())

## 7. Validação — Teste Ljung-Box

Verifica se os resíduos do modelo são ruído branco (ou seja, se o modelo já capturou toda a informação disponível na série). Buscamos p-value **acima de 0,05**.

In [ ]:
residuos = modelo_treinado.resid
resultado_teste = acorr_ljungbox(residuos, lags=[7])
resultado_teste

## 8. Previsão para os próximos 7 dias

In [ ]:
previsao_7dias = modelo_treinado.get_forecast(steps=7)

previsao_media = previsao_7dias.predicted_mean
conf_int = previsao_7dias.conf_int(alpha=0.05)  # 95% de confiança

tabela_previsao = pd.DataFrame({
    'previsao_venda': previsao_media,
    'limite_inferior_95': conf_int.iloc[:, 0],
    'limite_superior_95': conf_int.iloc[:, 1]
})
tabela_previsao

In [ ]:
# Total previsto para a semana
print(f"Total previsto para os próximos 7 dias: R$ {tabela_previsao['previsao_venda'].sum():,.2f}")

In [ ]:
# Plot: histórico (últimos 60 dias) + previsão dos próximos 7 dias
plt.figure(figsize=(14, 6))

historico_recente = df['venda'].iloc[-60:]
plt.plot(historico_recente.index, historico_recente, label='Histórico (últimos 60 dias)')

plt.plot(tabela_previsao.index, tabela_previsao['previsao_venda'],
         label='Previsão (7 dias)', color='red', marker='o')

plt.fill_between(tabela_previsao.index,
                  tabela_previsao['limite_inferior_95'],
                  tabela_previsao['limite_superior_95'],
                  color='pink', alpha=0.3, label='Intervalo de Confiança 95%')

plt.title('Previsão de Vendas Diárias — Próximos 7 Dias (Modelo SARIMAX Semanal)')
plt.xlabel('Data')
plt.ylabel('Venda (R$)')
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Notas

- O modelo usa sazonalidade **semanal** (período = 7), diferente do modelo mensal original (período = 12).
- A escolha da ordem não-sazonal `(p, d, q)` é feita automaticamente pelo menor AIC entre os candidatos testados — adicione mais combinações na lista `candidatos` se quiser testar outras variações.
- Se o teste Ljung-Box (seção 7) apontar p-value abaixo de 0,05, os resíduos ainda têm padrão não capturado — vale testar outras ordens ou incluir variáveis exógenas (ex: feriados, dia da semana como dummy).
- Este notebook prevê a coluna `venda` (valor bruto). Para prever `valor_total` (líquido) ou `atendimentos`, basta trocar a coluna-alvo nas seções 3 em diante.